# FarmTech Solutions — Fase 6 (Entrega 2)
## Comparação: YOLO adaptada vs YOLO padrão vs CNN do zero

**FIAP — Curso de Inteligência Artificial**

**Grupo:**
- Karina Garta Szewczuk — RM569309
- Maria Sabrina Feitosa da Silva — RM568714
- Nicolas Lima Apolinário — RM570741
- Roger Gabriel de Souza Jesus Costa — RM573659

---

Agora que já treinamos a YOLO na Entrega 1, a ideia aqui é pegar a mesma base de imagens e testar com três abordagens diferentes pra poder comparar:

- **YOLO adaptada** — o modelo que treinamos na E1, que conhece nossos objetos
- **YOLO padrão** — a YOLOv5 pré-treinada no COCO (80 classes), sem treino nenhum nosso
- **CNN do zero** — uma rede convolucional simples, construída no Keras, que classifica a imagem inteira (sem localizar o objeto)

No final monta uma tabela comparando as três em precisão, tempo de treino e tempo de inferência.

Lembrar de rodar no Colab com GPU ligada e o Drive montado.


## 1. Drive e ambiente

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -qr requirements.txt

import torch
print('GPU:', torch.cuda.is_available())


## 2. Configuração

Mesma base da E1. O `best.pt` é o modelo que salvamos no Drive no final da Entrega 1.

In [ ]:
import os

BASE = '/content/drive/MyDrive/farmtech_fase6/dataset'
RAIZ = os.path.dirname(BASE)

# modelo treinado na E1
BEST_PT = f'{RAIZ}/best.pt'

CLASSE_A = 'objeto_a'   # trocar pelo nome real
CLASSE_B = 'objeto_b'   # trocar pelo nome real

IMG_SIZE = 640
print('Modelo:', BEST_PT, '| existe:', os.path.exists(BEST_PT))


In [ ]:
import yaml

config = {
    'train': f'{BASE}/images/train',
    'val':   f'{BASE}/images/val',
    'nc': 2,
    'names': [CLASSE_A, CLASSE_B],
}
with open('data.yaml', 'w') as f:
    yaml.dump(config, f, sort_keys=False)
print(open('data.yaml').read())


## 3. YOLO adaptada (modelo da E1)

Rodando o nosso modelo treinado pra medir a precisão e o tempo de inferência.

In [ ]:
# mAP e precision na validação
!python val.py --weights {BEST_PT} --data data.yaml --img {IMG_SIZE} --name val_adaptada


In [ ]:
import time

# tempo de inferência no teste
t0 = time.time()
!python detect.py --weights {BEST_PT} --source {BASE}/images/test --img {IMG_SIZE} --conf 0.25 --name det_adaptada
tempo_yolo_adaptada = time.time() - t0
print(f'\nTempo de inferência: {tempo_yolo_adaptada:.1f} s')


## 4. YOLO padrão (COCO, sem treino nosso)

A yolov5s.pt vem treinada nas 80 classes do COCO (pessoa, carro, cachorro...). A gente roda ela nas mesmas imagens de teste, sem nenhum treino. Se nossos objetos não forem dessas 80 classes, ela não vai detectar — e é exatamente esse o ponto da comparação.

In [ ]:
import time

t0 = time.time()
!python detect.py --weights yolov5s.pt --source {BASE}/images/test --img {IMG_SIZE} --conf 0.25 --name det_padrao
tempo_yolo_padrao = time.time() - t0
print(f'\nTempo de inferência: {tempo_yolo_padrao:.1f} s')


In [ ]:
from IPython.display import Image, display
import glob

# mostra umas detecções pra ver o que ela achou
for caminho in sorted(glob.glob('runs/detect/det_padrao/*.jpg'))[:4]:
    display(Image(filename=caminho, width=450))


Se os objetos não são do COCO, ela não detecta (ou detecta como outra coisa). É super fácil de usar, mas não serve pra objetos específicos.

## 5. CNN treinada do zero

Aqui muda a abordagem: em vez de detectar (achar onde está), a CNN classifica a imagem inteira — é classe A ou classe B. Montamos uma rede simples no Keras e treinamos com as mesmas imagens.

Primeiro precisa reorganizar as imagens em pastas por classe, porque é assim que o Keras espera.

### 5.1 Reorganizar por classe

In [ ]:
import os, glob, shutil

CLASSIF = '/content/classificacao'
classes = {0: CLASSE_A, 1: CLASSE_B}

def organiza(split):
    for img in glob.glob(f'{BASE}/images/{split}/*'):
        nome = os.path.splitext(os.path.basename(img))[0]
        rotulo = f'{BASE}/labels/{split}/{nome}.txt'
        if not os.path.exists(rotulo):
            continue
        with open(rotulo) as f:
            linha = f.readline().split()
        if not linha:
            continue
        cls_id = int(linha[0])
        dest = f'{CLASSIF}/{split}/{classes[cls_id]}'
        os.makedirs(dest, exist_ok=True)
        shutil.copy(img, dest)

for split in ['train', 'val']:
    organiza(split)

for split in ['train', 'val']:
    for c in classes.values():
        pasta = f'{CLASSIF}/{split}/{c}'
        qtd = len(glob.glob(f'{pasta}/*')) if os.path.exists(pasta) else 0
        print(f'{split}/{c}: {qtd}')


### 5.2 Montar a CNN

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

IMG = (160, 160)
BATCH = 8

train_ds = tf.keras.utils.image_dataset_from_directory(
    f'{CLASSIF}/train', image_size=IMG, batch_size=BATCH, label_mode='binary')
val_ds = tf.keras.utils.image_dataset_from_directory(
    f'{CLASSIF}/val', image_size=IMG, batch_size=BATCH, label_mode='binary')

print('Classes:', train_ds.class_names)


In [ ]:
# data augmentation pra base pequena
aug = models.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# 3 blocos de convolução + classificador binário
cnn = models.Sequential([
    layers.Input((160, 160, 3)),
    aug,
    layers.Rescaling(1./255),
    layers.Conv2D(16, 3, activation='relu'), layers.MaxPooling2D(),
    layers.Conv2D(32, 3, activation='relu'), layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'), layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid'),
])

cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn.summary()


### 5.3 Treinar

In [ ]:
import time

t0 = time.time()
hist = cnn.fit(train_ds, validation_data=val_ds, epochs=20)
tempo_treino_cnn = time.time() - t0
print(f'\nTempo de treino: {tempo_treino_cnn:.1f} s')


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(hist.history['accuracy'], marker='o', label='treino')
axes[0].plot(hist.history['val_accuracy'], marker='o', label='validação')
axes[0].set_title('Acurácia'); axes[0].set_xlabel('época'); axes[0].legend()

axes[1].plot(hist.history['loss'], marker='o', label='treino')
axes[1].plot(hist.history['val_loss'], marker='o', label='validação')
axes[1].set_title('Perda'); axes[1].set_xlabel('época'); axes[1].legend()

plt.tight_layout()
plt.savefig('/content/cnn_curvas.png', dpi=120)
plt.show()

acc_cnn = hist.history['val_accuracy'][-1]
print(f'Acurácia na validação: {acc_cnn:.2%}')


In [ ]:
import time, numpy as np

# tempo de inferência por imagem
for imgs, _ in val_ds.take(1):
    t0 = time.time()
    _ = cnn.predict(imgs, verbose=0)
    tempo_inf_cnn = (time.time() - t0) / len(imgs)
print(f'Inferência: {tempo_inf_cnn*1000:.1f} ms por imagem')


Com só 64 imagens de treino é normal a CNN ter dificuldade. Se a acurácia de treino fica alta mas a de validação oscila, é overfitting — faz sentido, porque sem transfer learning a rede precisa de muito mais dado pra aprender.

## 6. Comparação das três abordagens

*(preencher com os números que apareceram acima)*

| Critério | YOLO adaptada | YOLO padrão | CNN do zero |
| --- | --- | --- | --- |
| O que faz | detecta e localiza nossos objetos | detecta 80 classes genéricas | classifica a imagem inteira |
| Facilidade | média (rotular + treinar) | alta (zero treino) | média/baixa (montar a rede) |
| Precisão | *(mAP da seção 3)* | baixa nos nossos objetos | *(acurácia da seção 5.3)* |
| Tempo de treino | minutos | zero | *(seção 5.3)* |
| Tempo de inferência | *(seção 3)* | *(seção 4)* | *(seção 5.3)* |

A YOLO padrão é a mais fácil de usar mas só serve se os objetos forem do COCO — pro nosso caso não funciona. A YOLO adaptada é a melhor opção porque combina transfer learning com detecção dos nossos objetos específicos. A CNN do zero funciona pra classificação mas precisa de mais dados pra ter boa precisão, e não localiza o objeto na imagem.

No fim, a escolha depende do cenário: se o objeto é comum, a YOLO padrão resolve na hora. Se é específico e precisa localizar, a YOLO adaptada compensa o trabalho de rotulação. A CNN do zero é boa pra entender como a rede funciona, mas pra base pequena o transfer learning ganha.
